# Урок 01 — Вступ до AI агентів

Ласкаво просимо на перший урок курсу **AI агенти для початківців**!

**AI агент** — це програма, що використовує велику мовну модель (LLM) як свій механізм міркування і може здійснювати *дії* у реальному світі — викликати API, робити запити до баз даних або запускати код — щоб досягти мети від імені користувача.

У цьому зошиті ви створите свого першого агента: **туристичного агента**, який рекомендує місця для відпочинку. В процесі ви навчитеся:

1. Підключатися до Microsoft Foundry Agent Service за допомогою **Microsoft Agent Framework**.
2. Надати агенту **інструмент** — просту функцію на Python, яку він може викликати.
3. Запустити агента та проаналізувати його відповідь.
4. Потоково отримувати відповідь агента токен за токеном.


## Setup

Before running this notebook, make sure you have:

1. **A Microsoft Foundry project** with a deployed chat model (e.g. `gpt-4.1-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Створення вашого першого агента

Агенту потрібні дві речі:

- **Інструкції**, які вказують йому *хто він* і *як поводитись* (системний запит).
- **Інструменти** — Python-функції, задекоровані за допомогою `@tool`, які агент може викликати для отримання інформації або виконання дій.

Нижче ми визначаємо простий інструмент, який повертає список популярних місць для відпочинку. Агент використовуватиме цей інструмент, коли користувач попросить рекомендації щодо подорожей.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Потокові відповіді

Для більш інтерактивного досвіду ви можете **потоково** отримувати відповідь агента. Замість того, щоб чекати повної відповіді, агент видає текстові фрагменти по мірі їх генерації. Це особливо корисно в інтерфейсах чату, де потрібно відображати результати в реальному часі.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Підсумок

У цьому уроці ви дізналися, як:

- **Створити провайдера**, який підключається до Microsoft Foundry Agent Service через `FoundryChatClient`.
- **Визначити інструмент** за допомогою декоратора `@tool`, щоб агент міг викликати ваші функції на Python.
- **Запустити агента** з повідомленням користувача та вивести його відповідь.
- **Потоково отримувати відповіді** для виводу в режимі реального часу.

У наступному уроці ми детальніше розглянемо агентські фреймворки та навчимося надавати агентам більш потужні інструменти та можливості багатокрокового мислення.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
